# 00 — Data Collection
## BRFSS 2022 / 2023 / 2024 — ASCII to CSV Conversion

**Purpose**: Extract diabetes-related variables from CDC official ASC files using HTML codebooks and save as CSV files ready for analysis.

| Year | ASC File | Codebook |
|------|----------|----------|
| 2022 | LLCP2022.ASC | USCODE22_LLCP_102523.HTML |
| 2023 | LLCP2023.ASC | USCODE23_LLCP_021924.HTML |
| 2024 | LLCP2024.ASC | USCODE24_LLCP_082125.HTML |

**Output**: `data/raw/prepre/`


In [19]:
import re
import os
import pandas as pd
from pathlib import Path

print("pandas version :", pd.__version__)
print("Working directory:", os.getcwd())


pandas version : 2.0.3
Working directory: d:\Users\kotae\Documents\Portfolio\project\Project 2\diabetes-risk-prediction\notebooks


## 1. Path Configuration

Set the folder containing the raw ASC / HTML files and the output destination.


In [20]:
# ─────────────────────────────────────────────────
# Adjust these paths to match your local environment
# ─────────────────────────────────────────────────

# Folder containing the ASC and HTML codebook files
SOURCE_DIR = Path("../data/source")   # <- change if needed

# Output folder for processed CSV files
OUTPUT_DIR = Path("../data/raw")

# ─────────────────────────────────────────────────

# Year-to-file mapping
YEAR_FILES = {
    2022: {
        "asc":           SOURCE_DIR / "LLCP2022.ASC",
        "html":          SOURCE_DIR / "USCODE22_LLCP_102523.HTML",
        "record_length": 2052,
    },
    2023: {
        "asc":           SOURCE_DIR / "LLCP2023.ASC",
        "html":          SOURCE_DIR / "USCODE23_LLCP_021924.HTML",
        "record_length": 2111,
    },
    2024: {
        "asc":           SOURCE_DIR / "LLCP2024.ASC",
        "html":          SOURCE_DIR / "USCODE24_LLCP_082125.HTML",
        "record_length": 2111,
    },
}

# Create output directory if it does not exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory : {OUTPUT_DIR.resolve()}")


Output directory : D:\Users\kotae\Documents\Portfolio\project\Project 2\diabetes-risk-prediction\data\raw


## 2. Target Variable Definitions

Only the variable **names** are defined here.  
Column positions are extracted automatically from each year's HTML codebook.


In [21]:
# Target variables and their descriptions
TARGET_VARIABLES = {
    # --- Target variable ---
    "DIABETE4":  "Ever told you had diabetes (1=Yes, 3=No, 4=Pre-diabetes)",
    "PREDIAB2":  "Pre-diabetes or borderline diabetes confirmation",

    # --- Physical health ---
    "GENHLTH":   "General health status (1=Excellent to 5=Poor)",
    "PHYSHLTH":  "Days physical health not good (past 30 days)",
    "MENTHLTH":  "Days mental health not good (past 30 days)",
    "POORHLTH":  "Days poor health limited usual activities",
    "CHECKUP1":  "Time since last routine checkup",

    # --- Lifestyle ---
    "EXERANY2":  "Any physical activity in past 30 days (1=Yes, 2=No)",
    "DIFFWALK":  "Difficulty walking or climbing stairs (1=Yes, 2=No)",

    # --- Medical history ---
    "BPHIGH6":   "Ever told blood pressure was high (1=Yes, 3=No)",
    "CVDINFR4":  "Ever told had a heart attack (1=Yes, 2=No)",
    "CVDSTRK3":  "Ever told had a stroke (1=Yes, 2=No)",

    # --- Demographics ---
    "SEXVAR":    "Sex of respondent (1=Male, 2=Female)",
    "EDUCA":     "Highest education level (1=No school to 6=College graduate)",
    "INCOME3":   "Annual household income (1=<$10K to 11=$200K+)",
    "_STATE":    "State FIPS code",

    # --- Calculated variables ---
    "_SEX":      "Sex (calculated)",
    "_RACE":     "Race/ethnicity (calculated)",
    "_AGEG5YR":  "Age group in 5-year intervals (calculated)",
    "_BMI5CAT":  "BMI category (calculated)",
    "_SMOKER3":  "Smoking status (calculated)",
    "_CHOLCH3":  "Cholesterol check within past 5 years (calculated)",
}

print(f"Total target variables: {len(TARGET_VARIABLES)}")
print()
for name, desc in TARGET_VARIABLES.items():
    print(f"  {name:<15}  {desc}")


Total target variables: 22

  DIABETE4         Ever told you had diabetes (1=Yes, 3=No, 4=Pre-diabetes)
  PREDIAB2         Pre-diabetes or borderline diabetes confirmation
  GENHLTH          General health status (1=Excellent to 5=Poor)
  PHYSHLTH         Days physical health not good (past 30 days)
  MENTHLTH         Days mental health not good (past 30 days)
  POORHLTH         Days poor health limited usual activities
  CHECKUP1         Time since last routine checkup
  EXERANY2         Any physical activity in past 30 days (1=Yes, 2=No)
  DIFFWALK         Difficulty walking or climbing stairs (1=Yes, 2=No)
  BPHIGH6          Ever told blood pressure was high (1=Yes, 3=No)
  CVDINFR4         Ever told had a heart attack (1=Yes, 2=No)
  CVDSTRK3         Ever told had a stroke (1=Yes, 2=No)
  SEXVAR           Sex of respondent (1=Male, 2=Female)
  EDUCA            Highest education level (1=No school to 6=College graduate)
  INCOME3          Annual household income (1=<$10K to 11=$200K

## 3. Codebook Parser

Parses each year's HTML codebook to extract variable names and their starting column positions.


In [22]:
def parse_codebook(html_path: Path) -> dict:
    """
    Parse an HTML BRFSS codebook and return a dict of {variable_name: column_number}.
    Column numbers are 1-indexed as specified in the codebook.
    """
    with open(html_path, encoding="windows-1252") as f:
        content = f.read()

    pattern = r"Column:&nbsp;(\d+).*?SAS&nbsp;Variable&nbsp;Name:&nbsp;([A-Z_][A-Z0-9_]*)"
    matches = re.findall(pattern, content)
    return {name: int(col) for col, name in matches}


def build_colspecs(var_col: dict, target_vars: dict) -> tuple:
    """
    Build colspecs and column name list for pd.read_fwf().
    Column width is estimated from the gap to the next variable (capped at 10).
    """
    sorted_all = sorted(var_col.items(), key=lambda x: x[1])
    col_to_width = {}
    for i, (name, col) in enumerate(sorted_all):
        if i + 1 < len(sorted_all):
            col_to_width[name] = sorted_all[i + 1][1] - col
        else:
            col_to_width[name] = 2

    names, colspecs, missing = [], [], []

    for var in target_vars:
        if var not in var_col:
            missing.append(var)
            continue
        start = var_col[var] - 1                       # convert to 0-indexed
        width = min(col_to_width.get(var, 2), 10)
        names.append(var)
        colspecs.append((start, start + width))

    if missing:
        print(f"  WARNING — variables not found in codebook: {missing}")

    return names, colspecs


# Verify with 2023 codebook
print("── 2023 Codebook Verification ──")
var_col_2023 = parse_codebook(YEAR_FILES[2023]["html"])
print(f"  Total variables detected: {len(var_col_2023)}")

names_test, colspecs_test = build_colspecs(var_col_2023, TARGET_VARIABLES)
print(f"\n  {'Variable':<15} {'Start':>7}  {'End':>7}")
print("  " + "-" * 32)
for name, (s, e) in zip(names_test, colspecs_test):
    print(f"  {name:<15} {s:>7}  {e:>7}")


── 2023 Codebook Verification ──
  Total variables detected: 344

  Variable          Start      End
  --------------------------------
  DIABETE4            148      149
  PREDIAB2            260      261
  GENHLTH             100      101
  PHYSHLTH            101      103
  MENTHLTH            103      105
  POORHLTH            105      107
  CHECKUP1            111      112
  EXERANY2            112      113
  DIFFWALK            217      218
  BPHIGH6             132      133
  CVDINFR4            137      138
  CVDSTRK3            139      140
  SEXVAR               87       97
  EDUCA               186      187
  INCOME3             203      205
  _STATE                0       10
  _SEX               2063     2064
  _RACE              2059     2060
  _AGEG5YR           2064     2066
  _BMI5CAT           2085     2086
  _SMOKER3           2090     2091
  _CHOLCH3           1991     1992


## 4. ASC File Loader


In [23]:
def load_brfss_asc(year: int, year_config: dict, target_vars: dict) -> pd.DataFrame:
    """
    Load one year of BRFSS ASC data.
    Steps:
      1. Parse the HTML codebook to get column positions.
      2. Build colspecs for pd.read_fwf().
      3. Read the fixed-width ASC file.
      4. Convert columns to numeric.
      5. Prepend a YEAR column.
    """
    asc_path  = year_config["asc"]
    html_path = year_config["html"]

    print(f"\n{'=' * 55}")
    print(f"  Processing {year} data")
    print(f"{'=' * 55}")

    # Step 1 — Parse codebook
    print(f"  [1/3] Parsing codebook ... {html_path.name}")
    var_col = parse_codebook(html_path)
    print(f"        {len(var_col)} variables detected")

    # Step 2 — Build colspecs
    names, colspecs = build_colspecs(var_col, target_vars)
    print(f"  [2/3] Variables to extract: {len(names)}")

    # Step 3 — Read ASC file
    file_mb = asc_path.stat().st_size / 1024 / 1024
    print(f"  [3/3] Reading ASC file ... {asc_path.name} ({file_mb:.0f} MB)")

    df = pd.read_fwf(
        asc_path,
        colspecs=colspecs,
        names=names,
        dtype=str,
        encoding="latin-1",
    )

    # Convert to numeric (blanks and invalid codes become NaN)
    for col in df.columns:
        df[col] = pd.to_numeric(df[col].str.strip(), errors="coerce")

    # Add YEAR column
    df.insert(0, "YEAR", year)

    print(f"  Done: {df.shape[0]:,} rows x {df.shape[1]} columns")
    return df


## 5. Process All Three Years and Save


In [24]:
all_years = {}

for year, config in YEAR_FILES.items():
    df = load_brfss_asc(year, config, TARGET_VARIABLES)
    all_years[year] = df

    # Save individual year CSV
    out_path = OUTPUT_DIR / f"brfss_{year}_diabetes.csv"
    df.to_csv(out_path, index=False)
    size_mb = out_path.stat().st_size / 1024 / 1024
    print(f"  Saved: {out_path.name} ({size_mb:.1f} MB)")

print("\nAll years processed successfully.")



  Processing 2022 data
  [1/3] Parsing codebook ... USCODE22_LLCP_102523.HTML
        324 variables detected
  WARNING — variables not found in codebook: ['BPHIGH6', '_RACE', '_CHOLCH3']
  [2/3] Variables to extract: 19
  [3/3] Reading ASC file ... LLCP2022.ASC (872 MB)
  Done: 445,132 rows x 20 columns
  Saved: brfss_2022_diabetes.csv (30.5 MB)

  Processing 2023 data
  [1/3] Parsing codebook ... USCODE23_LLCP_021924.HTML
        344 variables detected
  [2/3] Variables to extract: 22
  [3/3] Reading ASC file ... LLCP2023.ASC (873 MB)
  Done: 433,323 rows x 23 columns
  Saved: brfss_2023_diabetes.csv (33.9 MB)

  Processing 2024 data
  [1/3] Parsing codebook ... USCODE24_LLCP_082125.HTML
        297 variables detected
  WARNING — variables not found in codebook: ['BPHIGH6', '_CHOLCH3']
  [2/3] Variables to extract: 20
  [3/3] Reading ASC file ... LLCP2024.ASC (900 MB)
  Done: 457,670 rows x 21 columns
  Saved: brfss_2024_diabetes.csv (32.3 MB)

All years processed successfully.


## 6. Create Combined Dataset (2022–2024)


In [25]:
df_all = pd.concat(all_years.values(), ignore_index=True)

print(f"Combined dataset: {df_all.shape[0]:,} rows x {df_all.shape[1]} columns")
print()
print("Row count by year:")
print(df_all["YEAR"].value_counts().sort_index().to_string())

# Save combined CSV
combined_path = OUTPUT_DIR / "brfss_2022_2024_combined.csv"
df_all.to_csv(combined_path, index=False)
size_mb = combined_path.stat().st_size / 1024 / 1024
print(f"\nSaved: {combined_path.name} ({size_mb:.1f} MB)")


Combined dataset: 1,336,125 rows x 23 columns

Row count by year:
YEAR
2022    445132
2023    433323
2024    457670

Saved: brfss_2022_2024_combined.csv (100.5 MB)


## 7. Quick Validation

Verify that the data loaded correctly before moving to the next phase.


In [26]:
# Distribution of DIABETE4 (target variable) by year
print("── DIABETE4 Distribution by Year ──")
print("Value meanings:")
print("  1 = Yes (diabetes)")
print("  2 = Yes, but only during pregnancy (female)")
print("  3 = No")
print("  4 = No, pre-diabetes / borderline diabetes")
print("  7 = Don't know / Not sure")
print("  9 = Refused")
print()

for year, df in all_years.items():
    print(f"[{year}]")
    counts = df["DIABETE4"].value_counts(dropna=False).sort_index()
    total  = len(df)
    for val, cnt in counts.items():
        print(f"  {str(val):>5} : {cnt:>8,}  ({cnt / total * 100:5.1f}%)")
    print()


── DIABETE4 Distribution by Year ──
Value meanings:
  1 = Yes (diabetes)
  2 = Yes, but only during pregnancy (female)
  3 = No
  4 = No, pre-diabetes / borderline diabetes
  7 = Don't know / Not sure
  9 = Refused

[2022]
    2.0 :    3,836  (  0.9%)
    3.0 :  368,721  ( 82.8%)
    4.0 :   10,329  (  2.3%)
    7.0 :      763  (  0.2%)
    9.0 :      321  (  0.1%)
  101.0 :       58  (  0.0%)
  102.0 :       40  (  0.0%)
  103.0 :       46  (  0.0%)
  104.0 :       65  (  0.0%)
  105.0 :       85  (  0.0%)
  106.0 :       73  (  0.0%)
  107.0 :       96  (  0.0%)
  108.0 :      101  (  0.0%)
  109.0 :       88  (  0.0%)
  110.0 :      128  (  0.0%)
  111.0 :       91  (  0.0%)
  112.0 :      144  (  0.0%)
  113.0 :      115  (  0.0%)
  114.0 :      129  (  0.0%)
  115.0 :      156  (  0.0%)
  116.0 :      149  (  0.0%)
  117.0 :      170  (  0.0%)
  118.0 :      215  (  0.0%)
  119.0 :      155  (  0.0%)
  120.0 :      341  (  0.1%)
  121.0 :      211  (  0.0%)
  122.0 :      237  (  

In [27]:
# Missing value summary for 2023
print("── Missing Value Summary (2023) ──")
df23     = all_years[2023]
missing  = df23.isnull().sum()
miss_pct = (missing / len(df23) * 100).round(1)

summary = pd.DataFrame({
    "Missing Count": missing,
    "Missing (%)":   miss_pct,
})
print(summary.to_string())


── Missing Value Summary (2023) ──
          Missing Count  Missing (%)
YEAR                  0          0.0
DIABETE4              5          0.0
PREDIAB2         267171         61.7
GENHLTH               4          0.0
PHYSHLTH              3          0.0
MENTHLTH              3          0.0
POORHLTH         181153         41.8
CHECKUP1              2          0.0
EXERANY2              2          0.0
DIFFWALK          16146          3.7
BPHIGH6               3          0.0
CVDINFR4              3          0.0
CVDSTRK3              4          0.0
SEXVAR                0          0.0
EDUCA                 9          0.0
INCOME3            8075          1.9
_STATE                0          0.0
_SEX                  0          0.0
_RACE                86          0.0
_AGEG5YR              0          0.0
_BMI5CAT          40535          9.4
_SMOKER3              0          0.0
_CHOLCH3              0          0.0


In [28]:
# Preview first 5 rows of the combined dataset
print("── Combined Dataset — First 5 Rows ──")
df_all.head()


── Combined Dataset — First 5 Rows ──


,YEAR,DIABETE4,PREDIAB2,GENHLTH,PHYSHLTH,MENTHLTH,POORHLTH,CHECKUP1,EXERANY2,DIFFWALK,...,EDUCA,INCOME3,_STATE,_SEX,_AGEG5YR,_BMI5CAT,_SMOKER3,BPHIGH6,_RACE,_CHOLCH3
0,2022,180.0,NaN,2.0,88.0,88.0,NaN,1.0,2.0,2.0,...,6.0,99.0,1,2,13,NaN,4,NaN,NaN,NaN
1,2022,3.0,NaN,1.0,88.0,88.0,NaN,8.0,2.0,2.0,...,4.0,5.0,1,2,13,3.0,4,NaN,NaN,NaN
2,2022,3.0,NaN,2.0,2.0,3.0,2.0,1.0,1.0,2.0,...,6.0,10.0,1,2,8,3.0,4,NaN,NaN,NaN
3,2022,3.0,NaN,1.0,88.0,88.0,NaN,1.0,1.0,2.0,...,4.0,77.0,1,2,14,2.0,2,NaN,NaN,NaN
4,2022,3.0,NaN,4.0,2.0,88.0,88.0,1.0,1.0,2.0,...,5.0,5.0,1,2,5,2.0,4,NaN,NaN,NaN


## Summary

The following files have been saved to `data/raw/`:

| File | Contents |
|------|----------|
| `brfss_2022_diabetes.csv` | 2022 single-year data |
| `brfss_2023_diabetes.csv` | 2023 single-year data |
| `brfss_2024_diabetes.csv` | 2024 single-year data |
| `brfss_2022_2024_combined.csv` | **Combined 3-year dataset (primary file)** |